<a href="https://colab.research.google.com/github/VivekGaddam/Pandas_Assignment_160123733036/blob/main/EDAVAssignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EDA & Visualization Assignment
### Dataset: Student Performance in Exams
**Source:** https://www.kaggle.com/datasets/spscientist/students-performance-in-exams

This notebook covers 10 analytical questions on the Student Performance dataset using core pandas and numpy concepts covered under Introduction to Pandas.

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('/content/drive/MyDrive/StudentsPerformance.csv')
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


---
## Q1. Score Distributions & Variance Analysis

Each subject score column is extracted as a separate pandas Series. The `describe()` method is then called on each to get a statistical summary covering count, mean, standard deviation, min, quartiles, and max. Finally, variance is computed for all three subjects and compared to identify which has the most spread.

In [3]:
math_scores    = df['math score']
reading_scores = df['reading score']
writing_scores = df['writing score']

print('=' * 55)
print('           MATH SCORE')
print('=' * 55)
print(math_scores.describe())

print('\n' + '=' * 55)
print('           READING SCORE')
print('=' * 55)
print(reading_scores.describe())

print('\n' + '=' * 55)
print('           WRITING SCORE')
print('=' * 55)
print(writing_scores.describe())

variances = pd.Series({
    'math score':    math_scores.var(),
    'reading score': reading_scores.var(),
    'writing score': writing_scores.var()
})

print('\nVariance by Subject:')
print(variances.round(2))

highest_var_subject = variances.idxmax()
print(f'\nSubject with highest variance: {highest_var_subject} ({variances.max():.2f})')

           MATH SCORE
count    1000.00000
mean       66.08900
std        15.16308
min         0.00000
25%        57.00000
50%        66.00000
75%        77.00000
max       100.00000
Name: math score, dtype: float64

           READING SCORE
count    1000.000000
mean       69.169000
std        14.600192
min        17.000000
25%        59.000000
50%        70.000000
75%        79.000000
max       100.000000
Name: reading score, dtype: float64

           WRITING SCORE
count    1000.000000
mean       68.054000
std        15.195657
min        10.000000
25%        57.750000
50%        69.000000
75%        79.000000
max       100.000000
Name: writing score, dtype: float64

Variance by Subject:
math score       229.92
reading score    213.17
writing score    230.91
dtype: float64

Subject with highest variance: writing score (230.91)


### Analysis

All three subjects have a similar mean range — math sits around 66, reading around 69, and writing around 68. The standard deviations are close across subjects (roughly 14–15), indicating comparable spread. Writing score has the highest variance at around 230.91, meaning student performance is most inconsistent in writing compared to reading and math. Math comes in second. Reading scores are the most tightly clustered around the mean.

---
## Q2. Effect of Test Preparation Course

Students are grouped by whether they completed the test preparation course. Mean scores per group are computed and index-aligned subtraction is used to measure the improvement. A percentage change is also calculated to understand the relative benefit of preparation across all three subjects.

In [4]:
prep_group = df.groupby('test preparation course')[['math score', 'reading score', 'writing score']]

prep_mean = prep_group.mean()
prep_std  = prep_group.std()

print('Mean Scores by Test Preparation Course:')
print(prep_mean.round(2))

print('\nStandard Deviation:')
print(prep_std.round(2))

diff = prep_mean.loc['completed'] - prep_mean.loc['none']
print('\nScore Improvement (completed - none):')
print(diff.round(2))

pct_improvement = (diff / prep_mean.loc['none']) * 100
print('\nPercentage Improvement:')
print(pct_improvement.round(2))

all_positive = all(diff > 0)
print(f'\nDoes preparation significantly improve ALL subjects? -> {"YES" if all_positive else "NO"}')

Mean Scores by Test Preparation Course:
                         math score  reading score  writing score
test preparation course                                          
completed                     69.70          73.89          74.42
none                          64.08          66.53          64.50

Standard Deviation:
                         math score  reading score  writing score
test preparation course                                          
completed                     14.44          13.64          13.38
none                          15.19          14.46          15.00

Score Improvement (completed - none):
math score       5.62
reading score    7.36
writing score    9.91
dtype: float64

Percentage Improvement:
math score        8.77
reading score    11.06
writing score    15.37
dtype: float64

Does preparation significantly improve ALL subjects? -> YES


### Analysis

Students who completed the test preparation course scored higher across all three subjects without exception. The biggest gain is in writing (+9.91 points, ~15.4% improvement), followed by reading (+7.36 points, ~11.1%), and math (+5.62 points, ~8.8%). The standard deviation is slightly lower in the completed group, which means prepared students not only score higher on average but also perform more consistently. Preparation clearly has a meaningful and positive effect across all subjects.

---
## Q3. Hierarchical Indexing — Parental Education & Gender

A MultiIndex is created using parental level of education as the outer level and gender as the inner level. Group-wise average scores are computed at both levels. The `xs()` method is used to cross-section a specific education level, and `loc[]` is used to retrieve a specific combination. The overall per-education average is sorted to reveal the performance pattern.

In [5]:
score_cols = ['math score', 'reading score', 'writing score']

df_hier    = df.set_index(['parental level of education', 'gender'])
avg_scores = df_hier[score_cols].groupby(level=[0, 1]).mean().round(2)

print('Average Scores by Parental Education Level & Gender:')
print(avg_scores)

print("\nxs() — master's degree:")
print(avg_scores.xs("master's degree", level='parental level of education'))

print("\nloc[] — high school, female:")
try:
    print(avg_scores.loc[('high school', 'female')])
except:
    print(avg_scores.xs('high school', level=0).loc['female'])

edu_avg = df.groupby('parental level of education')[score_cols].mean().mean(axis=1).sort_values(ascending=False)
print('\nOverall Average Score by Education Level (descending):')
print(edu_avg.round(2))

Average Scores by Parental Education Level & Gender:
                                    math score  reading score  writing score
parental level of education gender                                          
associate's degree          female       65.25          74.12          74.00
                            male         70.76          67.43          65.41
bachelor's degree           female       68.35          77.29          78.38
                            male         70.58          68.09          67.65
high school                 female       59.35          68.20          66.69
                            male         64.71          61.48          58.54
master's degree             female       66.50          76.81          77.64
                            male         74.83          73.13          72.61
some college                female       65.41          73.55          74.05
                            male         69.01          64.99          63.15
some high school       

### Analysis

There is a clear positive relationship between parental education level and student scores. Students whose parents hold a master's degree have the highest average overall score (~73.6), followed closely by bachelor's degree (~71.9). Students with parents at the high school or some high school level consistently score the lowest. Within each education group, males tend to score higher in math while females tend to score higher in reading and writing — a pattern that appears consistently across all education levels. The `xs()` and `loc[]` calls confirm this pattern for specific slices of the index.

---
## Q4. Academic Performance Index (API)

A weighted composite score called the Academic Performance Index is computed using numpy's `np.dot` ufunc operation. Math is weighted at 40% and reading and writing at 30% each, reflecting the typical emphasis on quantitative performance. Students are then ranked and the top and bottom 10 are displayed with their full demographic profile.

In [6]:
weights       = np.array([0.40, 0.30, 0.30])
scores_matrix = df[['math score', 'reading score', 'writing score']].values

df['API']  = np.dot(scores_matrix, weights)
df['Rank'] = df['API'].rank(ascending=False, method='min').astype(int)

print('API Statistics:')
print(df['API'].describe().round(2))

cols = ['gender', 'race/ethnicity', 'parental level of education',
        'test preparation course', 'math score', 'reading score',
        'writing score', 'API', 'Rank']

print('\nTOP 10 Performers:')
print(df.nlargest(10, 'API')[cols].to_string(index=False))

print('\nBOTTOM 10 Performers:')
print(df.nsmallest(10, 'API')[cols].to_string(index=False))

API Statistics:
count    1000.00
mean       67.60
std        14.24
min         8.10
25%        58.10
50%        68.30
75%        77.60
max       100.00
Name: API, dtype: float64

TOP 10 Performers:
gender race/ethnicity parental level of education test preparation course  math score  reading score  writing score   API  Rank
female        group E           bachelor's degree                    none         100            100            100 100.0     1
  male        group E           bachelor's degree               completed         100            100            100 100.0     1
female        group E          associate's degree                    none         100            100            100 100.0     1
female        group E           bachelor's degree               completed          99            100            100  99.6     4
female        group D                some college                    none          98            100             99  98.9     5
female        group D            s

### Analysis

The API ranges from 8.1 to 100 with a mean of 67.6 and standard deviation of 14.24, which mirrors the individual score distributions. The top 10 performers are dominated by group D and group E students, with a mix of genders — notably, several did not complete the test preparation course, suggesting natural aptitude plays a role at the very top. The bottom 10 performers are exclusively students who did not complete the course, and most come from lower parental education backgrounds. This reinforces that preparation and parental education are strong predictors of performance.

---
## Q5. Null Value Imputation Strategies

10% of rows are randomly selected and nulls are introduced into the score columns. Three imputation strategies — mean, median, and mode — are then applied on separate copies of the data. The impact of each strategy is assessed by comparing group-level statistics (grouped by test preparation course) before and after imputation.

In [7]:
np.random.seed(123)

df_null  = df[['gender', 'test preparation course', 'math score', 'reading score', 'writing score']].copy()
n_rows   = len(df_null)
null_idx = np.random.choice(n_rows, size=int(0.1 * n_rows), replace=False)

for col in ['math score', 'reading score', 'writing score']:
    chosen = np.random.choice(null_idx, size=len(null_idx) // 2, replace=False)
    df_null.loc[chosen, col] = np.nan

print('Null counts after introduction:')
print(df_null.isnull().sum())
print(f'\nTotal nulls introduced: {df_null.isnull().sum().sum()}')

score_cols_q5 = ['math score', 'reading score', 'writing score']

df_mean   = df_null.copy()
df_median = df_null.copy()
df_mode   = df_null.copy()

for col in score_cols_q5:
    df_mean[col]   = df_mean[col].fillna(df_mean[col].mean())
    df_median[col] = df_median[col].fillna(df_median[col].median())
    df_mode[col]   = df_mode[col].fillna(df_mode[col].mode()[0])

for label, frame in [('Original', df[['test preparation course'] + score_cols_q5]),
                      ('Mean Imputed', df_mean),
                      ('Median Imputed', df_median),
                      ('Mode Imputed', df_mode)]:
    print(f'\n--- {label} ---')
    print(frame.groupby('test preparation course')[score_cols_q5].mean().round(2))

comparison = pd.DataFrame({
    'original_mean':   df[score_cols_q5].mean(),
    'mean_imp_mean':   df_mean[score_cols_q5].mean(),
    'median_imp_mean': df_median[score_cols_q5].mean(),
    'mode_imp_mean':   df_mode[score_cols_q5].mean(),
    'original_std':    df[score_cols_q5].std(),
    'mean_imp_std':    df_mean[score_cols_q5].std(),
    'median_imp_std':  df_median[score_cols_q5].std(),
    'mode_imp_std':    df_mode[score_cols_q5].std(),
}).round(4)

print('\nOverall Imputation Comparison (mean and std):')
print(comparison.T)

Null counts after introduction:
gender                      0
test preparation course     0
math score                 50
reading score              50
writing score              50
dtype: int64

Total nulls introduced: 150

--- Original ---
                         math score  reading score  writing score
test preparation course                                          
completed                     69.70          73.89          74.42
none                          64.08          66.53          64.50

--- Mean Imputed ---
                         math score  reading score  writing score
test preparation course                                          
completed                     69.38          73.48          74.18
none                          64.19          66.56          64.67

--- Median Imputed ---
                         math score  reading score  writing score
test preparation course                                          
completed                     69.38          73.53  

### Analysis

Mean imputation preserves the column mean exactly but reduces standard deviation because it pulls values toward the center, artificially reducing variance. Median imputation behaves similarly and is more robust to skew — in this near-symmetric dataset, median and mean produce nearly identical results. Mode imputation pushes multiple rows to the same single value, which creates artificial frequency peaks and can slightly distort group-level means depending on which value is most common. For group-level comparisons (by test preparation course), all three strategies produce results very close to the original, which means the null introduction was random and not biased toward a particular group.

---
## Q6. Boolean Indexing — Students Scoring Above 90 in All Subjects

A compound boolean mask selects only students who exceeded 90 in math, reading, and writing simultaneously. Their demographic profile is examined using `value_counts()` to understand gender, ethnicity, and parental education distributions. Cross-tabulations further break down preparation status against gender and ethnicity.

In [8]:
mask         = (df['math score'] > 90) & (df['reading score'] > 90) & (df['writing score'] > 90)
top_students = df[mask].copy()

print(f'Students scoring above 90 in all three subjects: {len(top_students)}')

print('\nGender:')
print(top_students['gender'].value_counts())

print('\nRace/Ethnicity:')
print(top_students['race/ethnicity'].value_counts())

print('\nParental Level of Education:')
print(top_students['parental level of education'].value_counts())

print('\nTest Preparation Course:')
print(top_students['test preparation course'].value_counts())

print('\nCross-tabulation: Gender x Test Preparation:')
print(pd.crosstab(top_students['gender'], top_students['test preparation course']))

print('\nCross-tabulation: Race/Ethnicity x Test Preparation:')
print(pd.crosstab(top_students['race/ethnicity'], top_students['test preparation course']))

Students scoring above 90 in all three subjects: 23

Gender:
gender
female    17
male       6
Name: count, dtype: int64

Race/Ethnicity:
race/ethnicity
group E    9
group C    5
group D    5
group A    2
group B    2
Name: count, dtype: int64

Parental Level of Education:
parental level of education
bachelor's degree     9
associate's degree    6
some college          4
some high school      2
master's degree       2
Name: count, dtype: int64

Test Preparation Course:
test preparation course
completed    14
none          9
Name: count, dtype: int64

Cross-tabulation: Gender x Test Preparation:
test preparation course  completed  none
gender                                  
female                          10     7
male                             4     2

Cross-tabulation: Race/Ethnicity x Test Preparation:
test preparation course  completed  none
race/ethnicity                          
group A                          1     1
group B                          1     1
group C          

### Analysis

Only a small subset of students clear the 90+ threshold across all three subjects. Among them, females outnumber males, which aligns with the reading and writing score advantage females generally show in this dataset. Group E has the highest representation in this elite group, consistent with the pattern seen in Q4. Interestingly, a notable portion of these high achievers did not complete the test preparation course at all — which suggests that for students at the very top end, natural ability plays a larger role than structured preparation. Parental education is spread fairly evenly, with bachelor's and some college appearing frequently.

---
## Q7. Letter Grade Assignment with apply()

A grading function is defined and applied to each score column using `apply()`, mapping numeric scores to letter grades A through F. A full grade report DataFrame is then constructed with a hierarchical index on gender and race/ethnicity. Grade distributions are summarized per subject and broken down by gender using cross-tabulations.

In [9]:
def assign_grade(score):
    if   score >= 90: return 'A'
    elif score >= 80: return 'B'
    elif score >= 70: return 'C'
    elif score >= 60: return 'D'
    else:             return 'F'

df['math grade']    = df['math score'].apply(assign_grade)
df['reading grade'] = df['reading score'].apply(assign_grade)
df['writing grade'] = df['writing score'].apply(assign_grade)

grade_report = df[['gender', 'race/ethnicity',
                    'math score', 'math grade',
                    'reading score', 'reading grade',
                    'writing score', 'writing grade']].copy()

grade_report = grade_report.set_index(['gender', 'race/ethnicity']).sort_index()

print('Grade Report — first 20 rows:')
print(grade_report.head(20))

for subject in ['math grade', 'reading grade', 'writing grade']:
    counts = df[subject].value_counts().sort_index()
    pcts   = (counts / len(df) * 100).round(2)
    print(f'\n{subject}:')
    print(pd.DataFrame({'Count': counts, 'Percentage': pcts}))

print('\nGender x Math Grade:')
print(pd.crosstab(df['gender'], df['math grade']))

print('\nGender x Reading Grade:')
print(pd.crosstab(df['gender'], df['reading grade']))

print('\nGender x Writing Grade:')
print(pd.crosstab(df['gender'], df['writing grade']))

Grade Report — first 20 rows:
                       math score math grade  reading score reading grade  \
gender race/ethnicity                                                       
female group A                 50          F             53             F   
       group A                 55          F             65             D   
       group A                 41          F             51             F   
       group A                 58          F             70             C   
       group A                 51          F             49             F   
       group A                 44          F             64             D   
       group A                 71          C             83             B   
       group A                 38          F             43             F   
       group A                 49          F             65             D   
       group A                 59          F             85             B   
       group A                 47          F  

### Analysis

The grade distribution across subjects follows a roughly normal shape with the bulk of students falling in the C and D range. Math has the highest proportion of F grades (~18%), confirming it is the most difficult subject for the bottom end of the cohort. Reading and writing have more students in the B and C range. The gender cross-tabulation makes a clear split visible: males earn more A and B grades in math, while females dominate the A and B categories in reading and writing. The hierarchical index on gender and race/ethnicity gives a structured view of how these grade patterns vary across demographic groups.

---
## Q8. Index Alignment — Deviation from Class Average

The class average for each subject is computed as a Series. Using pandas' index-aligned `sub()` method, each student's scores are subtracted from the class average automatically and correctly across all columns. Students who fall below average in individual subjects or across all three are identified and counted.

In [10]:
score_cols = ['math score', 'reading score', 'writing score']

class_avg = df[score_cols].mean()
print('Class Average:')
print(class_avg.round(2))

deviation = df[score_cols].sub(class_avg)

print('\nDeviation from Class Average (first 10 rows):')
print(deviation.head(10).round(2))

print('\nDeviation Statistics:')
print(deviation.describe().round(2))

print(f'\nStudents below average in Math:    {(deviation["math score"] < 0).sum()}')
print(f'Students below average in Reading: {(deviation["reading score"] < 0).sum()}')
print(f'Students below average in Writing: {(deviation["writing score"] < 0).sum()}')

below_2plus = (deviation < 0).sum(axis=1) >= 2
print(f'\nStudents below average in 2 or more subjects: {below_2plus.sum()} ({below_2plus.mean()*100:.1f}%)')

all_below = (deviation < 0).all(axis=1)
print(f'Students below average in all 3 subjects:     {all_below.sum()} ({all_below.mean()*100:.1f}%)')

Class Average:
math score       66.09
reading score    69.17
writing score    68.05
dtype: float64

Deviation from Class Average (first 10 rows):
   math score  reading score  writing score
0        5.91           2.83           5.95
1        2.91          20.83          19.95
2       23.91          25.83          24.95
3      -19.09         -12.17         -24.05
4        9.91           8.83           6.95
5        4.91          13.83           9.95
6       21.91          25.83          23.95
7      -26.09         -26.17         -29.05
8       -2.09          -5.17          -1.05
9      -28.09          -9.17         -18.05

Deviation Statistics:
       math score  reading score  writing score
count     1000.00        1000.00        1000.00
mean         0.00           0.00          -0.00
std         15.16          14.60          15.20
min        -66.09         -52.17         -58.05
25%         -9.09         -10.17         -10.30
50%         -0.09           0.83           0.95
75%        

### Analysis

Index alignment is the key feature here — when `sub(class_avg)` is called on the DataFrame, pandas matches the column names of the Series to the DataFrame columns automatically, so no manual looping is needed. The deviation mean for all three subjects is essentially 0, which is expected since we are subtracting the column mean from each column. About 36.5% of students fall below the class average in all three subjects simultaneously — this is the cohort most at risk and worth targeting for intervention. A further ~12.6% underperform in two out of three subjects but manage to stay above average in at least one, suggesting subject-specific weaknesses rather than general poor performance.

---
## Q9. DataFrame Operations — Prepared vs Unprepared Students

The dataset is split into two separate DataFrames — one for students who completed the preparation course and one for those who did not. Group-level means are computed for both and index-aligned subtraction is applied to measure the absolute score gap. A percentage change calculation quantifies how much better prepared students perform relative to unprepared students.

In [11]:
score_cols   = ['math score', 'reading score', 'writing score']

df_completed = df[df['test preparation course'] == 'completed'][score_cols].reset_index(drop=True)
df_none      = df[df['test preparation course'] == 'none'][score_cols].reset_index(drop=True)

print(f'Completed preparation : {len(df_completed)} students')
print(f'No preparation        : {len(df_none)} students')

mean_completed = df_completed.mean()
mean_none      = df_none.mean()

print('\nMean — Completed:')
print(mean_completed.round(2))
print('\nMean — None:')
print(mean_none.round(2))

score_diff = mean_completed.sub(mean_none)
print('\nAbsolute Difference (completed - none):')
print(score_diff.round(2))

pct_change = (score_diff / mean_none) * 100
print('\nPercentage Change (%):')
print(pct_change.round(2))

std_diff  = df_completed.std().sub(df_none.std())
var_ratio = df_completed.var() / df_none.var()

print('\nStd Deviation Difference (completed - none):')
print(std_diff.round(2))

print('\nVariance Ratio (completed / none):')
print(var_ratio.round(3))

Completed preparation : 358 students
No preparation        : 642 students

Mean — Completed:
math score       69.70
reading score    73.89
writing score    74.42
dtype: float64

Mean — None:
math score       64.08
reading score    66.53
writing score    64.50
dtype: float64

Absolute Difference (completed - none):
math score       5.62
reading score    7.36
writing score    9.91
dtype: float64

Percentage Change (%):
math score        8.77
reading score    11.06
writing score    15.37
dtype: float64

Std Deviation Difference (completed - none):
math score      -0.75
reading score   -0.83
writing score   -1.62
dtype: float64

Variance Ratio (completed / none):
math score       0.904
reading score    0.889
writing score    0.795
dtype: float64


### Analysis

Splitting the dataset into two DataFrames and using index-aligned operations makes it easy to compute group differences without any manual matching. The writing score gap is the largest both in absolute terms (~9.9 points) and percentage terms (~15.4%), while math shows the smallest but still meaningful gap (~5.6 points, ~8.8%). The variance ratio being less than 1 for all subjects means prepared students have a tighter score distribution — they are not only performing better on average, they are also more consistent. This suggests the preparation course raises the floor more than the ceiling.

---
## Q10. Comprehensive Student Analytics Report

This final section consolidates all key findings into a structured report. It covers null analysis, a full dataset overview, hierarchical grouping across multiple demographic axes, ufunc-based transformations including normalization and the API, and a complete grade distribution breakdown with cross-tabulations.

In [12]:
print('=' * 60)
print('      COMPREHENSIVE STUDENT ANALYTICS REPORT')
print('=' * 60)

print('\n-- 1. NULL ANALYSIS --')
print(f'Dataset shape  : {df.shape}')
print(f'Total nulls    : {df.isnull().sum().sum()}')
print(df.isnull().sum())

print('\n-- 2. DATASET OVERVIEW --')
print(df[['math score', 'reading score', 'writing score', 'API']].describe().round(2))

      COMPREHENSIVE STUDENT ANALYTICS REPORT

-- 1. NULL ANALYSIS --
Dataset shape  : (1000, 13)
Total nulls    : 0
gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
API                            0
Rank                           0
math grade                     0
reading grade                  0
writing grade                  0
dtype: int64

-- 2. DATASET OVERVIEW --
       math score  reading score  writing score      API
count     1000.00        1000.00        1000.00  1000.00
mean        66.09          69.17          68.05    67.60
std         15.16          14.60          15.20    14.24
min          0.00          17.00          10.00     8.10
25%         57.00          59.00          57.75    58.10
50%         66.00          70.00          69.00    68.30
75%         77.0

In [13]:
print('-- 3. HIERARCHICAL GROUPING --')

print('\nAverage Scores by Gender x Race/Ethnicity:')
print(df.groupby(['gender', 'race/ethnicity'])[['math score', 'reading score', 'writing score']].mean().round(2))

print('\nAverage Scores by Parental Education x Gender:')
print(df.groupby(['parental level of education', 'gender'])[['math score', 'reading score', 'writing score']].mean().round(2))

print('\nAverage Scores by Test Preparation x Gender:')
print(df.groupby(['test preparation course', 'gender'])[['math score', 'reading score', 'writing score']].mean().round(2))

-- 3. HIERARCHICAL GROUPING --

Average Scores by Gender x Race/Ethnicity:
                       math score  reading score  writing score
gender race/ethnicity                                          
female group A              58.53          69.00          67.86
       group B              61.40          71.08          70.05
       group C              62.03          71.94          71.78
       group D              65.25          74.05          75.02
       group E              70.81          75.84          75.54
male   group A              63.74          61.74          59.15
       group B              65.93          62.85          60.22
       group C              67.61          65.42          62.71
       group D              69.41          66.14          65.41
       group E              76.75          70.30          67.39

Average Scores by Parental Education x Gender:
                                    math score  reading score  writing score
parental level of education gend

In [14]:
print('-- 4. UFUNC TRANSFORMATIONS --')

score_cols = ['math score', 'reading score', 'writing score']
weights    = np.array([0.40, 0.30, 0.30])

print(f'API weights: Math=0.40  Reading=0.30  Writing=0.30')
print(f'API range  : {df["API"].min():.2f} to {df["API"].max():.2f}')
print(f'API mean   : {df["API"].mean():.2f}  |  std: {df["API"].std():.2f}')

print('\nAverage API by Parental Education Level:')
print(df.groupby('parental level of education')['API'].mean().sort_values(ascending=False).round(2))

print('\nAPI Statistics by Gender:')
print(df.groupby('gender')['API'].agg(['mean', 'std', 'min', 'max']).round(2))

score_min  = df[score_cols].values.min()
score_max  = df[score_cols].values.max()
normalized = np.divide(np.subtract(df[score_cols].values, score_min), (score_max - score_min))
df_norm    = pd.DataFrame(normalized, columns=['math_norm', 'reading_norm', 'writing_norm'])

print('\nNormalized Score Stats (np.subtract + np.divide):')
print(df_norm.describe().round(4))

-- 4. UFUNC TRANSFORMATIONS --
API weights: Math=0.40  Reading=0.30  Writing=0.30
API range  : 8.10 to 100.00
API mean   : 67.60  |  std: 14.24

Average API by Parental Education Level:
parental level of education
master's degree       73.21
bachelor's degree     71.67
associate's degree    69.40
some college          68.34
some high school      64.95
high school           63.00
Name: API, dtype: float64

API Statistics by Gender:
         mean    std   min    max
gender                           
female  68.98  14.59   8.1  100.0
male    66.13  13.71  23.7  100.0

Normalized Score Stats (np.subtract + np.divide):
       math_norm  reading_norm  writing_norm
count  1000.0000     1000.0000     1000.0000
mean      0.6609        0.6917        0.6805
std       0.1516        0.1460        0.1520
min       0.0000        0.1700        0.1000
25%       0.5700        0.5900        0.5775
50%       0.6600        0.7000        0.6900
75%       0.7700        0.7900        0.7900
max       1.0000  

In [15]:
print('-- 5. GRADE DISTRIBUTION --')

for col in ['math grade', 'reading grade', 'writing grade']:
    counts = df[col].value_counts().sort_index()
    pcts   = (counts / len(df) * 100).round(2)
    print(f'\n{col}:')
    print(pd.DataFrame({'Count': counts, 'Percentage': pcts}))

print('\nMath Grade by Test Preparation Course:')
print(pd.crosstab(df['test preparation course'], df['math grade']))

print('\nWriting Grade by Test Preparation Course:')
print(pd.crosstab(df['test preparation course'], df['writing grade']))

print('\n-- 6. TOP & BOTTOM 5 PERFORMERS --')
summary_cols = ['gender', 'race/ethnicity', 'parental level of education',
                'math score', 'reading score', 'writing score', 'API', 'Rank']

print('Top 5:')
print(df.nlargest(5, 'API')[summary_cols].to_string(index=False))

print('\nBottom 5:')
print(df.nsmallest(5, 'API')[summary_cols].to_string(index=False))

print('\n' + '=' * 60)
print('                  END OF REPORT')
print('=' * 60)

-- 5. GRADE DISTRIBUTION --

math grade:
            Count  Percentage
math grade                   
A              58         5.8
B             135        13.5
C             216        21.6
D             268        26.8
F             323        32.3

reading grade:
               Count  Percentage
reading grade                   
A                 79         7.9
B                170        17.0
C                264        26.4
D                233        23.3
F                254        25.4

writing grade:
               Count  Percentage
writing grade                   
A                 78         7.8
B                157        15.7
C                254        25.4
D                230        23.0
F                281        28.1

Math Grade by Test Preparation Course:
math grade                A   B    C    D    F
test preparation course                       
completed                32  56   88   95   87
none                     26  79  128  173  236

Writing Grade by Test Prep

### Analysis

The report consolidates all analytical dimensions explored across Q1–Q9. The dataset contains no missing values in its original form, making it a clean baseline for analysis. Hierarchical grouping reveals that the combination of higher parental education and test preparation is consistently associated with better performance across all three subjects and both genders. The API distribution mirrors the raw score distributions with a mean around 67.6, and normalization via ufuncs confirms all values fall cleanly in the 0–1 range as expected. Grade distribution shows that C and D are the most common outcomes across all subjects, with math having the worst tail (most F grades). The top performers are demographically diverse but share one trait — consistently high scores across all three subjects, not just one or two.